# Run a small `ulens_lsst` simulation on DP0.2

This notebook uses the `ulens_lsst` v1.0.1 pipeline to simulate one PSPL microlensing event, inject it into DP0.2 images, perform forced photometry, and plot the ideal and measured light curves in all observed bands. The plot also shows the event magnification.

**You need:** the RSP **LSST** kernel and access to DP0.2. The repository includes the TRILEGAL and Genulens inputs used here, so this example does not require an AstroDataLab source query. Run the cells in order.

## 1. Clone and install `ulens_lsst`

The LSST Science Pipelines come from the RSP LSST kernel. This cell clones the tagged package if it is not already available in the current folder, then installs it in editable mode. If the package is already installed, you can skip this cell.

In [ ]:
from pathlib import Path
import os
import subprocess

repo = Path.cwd()
if not (repo / 'pyproject.toml').exists():
    repo = repo / 'ulens_lsst'
    if not (repo / 'pyproject.toml').exists():
        subprocess.run(
            ['git', 'clone', '--branch', 'v1.0.1', '--depth', '1',
             'https://github.com/nowokaren/ulens_lsst.git', str(repo)],
            check=True,
        )
os.chdir(repo)
%pip install -e .

## 2. Configure a small, reproducible run

This configuration uses one PSPL event, the bundled TRILEGAL/Genulens catalogs, and at most 40 DP0.2 exposures around the example field. The fixed event ID places the peak near an `i`-band visit for this field and exposure selection. The five configured bands avoid a `y`-band catalog issue in v1.0.0 and give observations in `ugriz`.

In [ ]:
from datetime import datetime
from pathlib import Path
import yaml
import ulens_lsst

package_root = Path(ulens_lsst.__file__).resolve().parent.parent
config = yaml.safe_load((package_root / 'config' / 'config_file.yaml').read_text())
config['name'] = f"ulens_lsst_pspl_demo_{datetime.now():%Y%m%d_%H%M%S}"
config['n_events'] = 10
config['nearby_object_count_radius_arcsec'] = 3.6
config['init_event_id'] = 1
config['model'] = 'PSPL'
config['system_type'] = 'BH'
config['sources_catalog'] = 'TRILEGAL'
config['bands'] = ['u', 'g', 'r', 'i', 'z', 'Y']
config['sky_center'] = {'coord': [62.0, -34.5], 'frame': 'icrs'}
config['sky_radius'] = 0.005
config['lsst_images']['data_preview'] = 'dp0'
config['lsst_images']['max_load_calexps'] = 60

tutorial_config = Path.cwd() / 'tutorial_config.yaml'
tutorial_config.write_text(yaml.safe_dump(config, sort_keys=False))
print(tutorial_config)

## 3. Create the pipeline

`SimPipeline` reads the YAML configuration and creates a new output folder under `runs/` for this run.

In [ ]:
from ulens_lsst.simulation_pipeline import SimPipeline

sim = SimPipeline(str(tutorial_config))
print(sim)

## 4. Simulate the event and ideal light curve

This step selects DP0.2 exposures in the configured region and generates the event parameters and ideal light curve. The table reports whether event generation succeeded.

In [ ]:
simulation_results = sim.simulate_lightcurves()
simulation_results

In [ ]:
print(simulation_results.error[0])

## 5. Find nearby sources

This updates the event catalog with the nearest Object and the number of unique Objects within the configured radius (`nearby_object_count_radius_arcsec`). The example uses 3.6 arcsec.

In [ ]:
sim.load_nearby_objects()

## 6. Inject into images and measure the source

This step injects the event into each selected exposure that covers its position, then performs forced photometry. Exposures that do not overlap the event can be skipped; check the returned status table.

In [ ]:
photometry_results = sim.process_synthetic_photometry()
photometry_results

## 7. Plot all observed bands and magnification

`Event.plot()` is the package's light-curve plotter: it shows the ideal model and measured photometry for each configured band with observations. PSPL magnification is achromatic, so the magnification points are overlaid once using the `i`-band epochs and a second y-axis.

In [ ]:
import matplotlib.pyplot as plt
from ulens_lsst.light_curves import Event
for i in range(10):
    event_id = int(sim.event_ids[i])
    event = Event.from_parquet(event_id, sim.calexps_photometry_file, sim.events_file)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    event.plot(
        bands=["u", "g", "r", "i", "z"],
        ax=ax,
        show_ideal=True,
        show_measured=True,
        show_baseline=True,
        show_params=True,
        show_nearby_object=True,
        show_times=True,
        show_legend=True,
        simulate_ideal=True,
    )
    
    i_band = event.photometry[event.photometry['band'] == 'i'].sort_values('time')
    ax2 = ax.twinx()
    ax2.plot(i_band['time'], i_band['magnification'], 'o', color='black', ms=3,
             label='Magnification A')
    ax2.set_ylabel('Magnification A')
    ax2.set_ylim(bottom=1)
    
    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    #ax.legend(lines + lines2, labels + labels2, loc='best', fontsize='small')
    fig.tight_layout()
    plt.show()

## 8. Inspect cutouts across bands

The existing `plot_event_fov()` function shows the light curve and one injected image cutout per band for a selected event. Each cutout marks the injected source, all catalog Objects inside the configured count radius, the nearest Object when present, and the radius circle.

In [ ]:
from ulens_lsst.analysis import plot_event_fov

selected_event_id = int(sim.event_ids[7])
plot_event_fov(
    event_id=selected_event_id,
    sim_name=sim.name,
    index_run=sim.new_file_index,
    bands=["u", "g", "r", "i", "z"],
    show_nearby_objects=False,
    save=False,
)

In [ ]:
event.photometry[event.photometry['band'] == 'i']

## 9. Sky map

This map shows the simulated event positions, catalog Objects, the injected region, and selected calexp footprints.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
from ulens_lsst.analysis import plot_sky_map

plot_sky_map(sim_name=sim.name, n_calexps=20, zoom=sim.sky_radius)
display(Image(filename=str(Path(sim.output_dir) / "analysis" / "sky_map.png")))